# Week 15, Day 5: Recursion versus loops

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w015d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
def total_rec(items):
    return 0 if not items else items[0] + total_rec(items[1:])

def total_loop(items):
    total = 0
    for x in items:
        total += x
    return total

data = list(range(100))
print(total_rec(data), total_loop(data))

In [ ]:
def factorial_rec(n):
    if n == 0:
        return 1
    return n * factorial_rec(n - 1)

def factorial_loop(n):
    result = 1
    while n > 0:
        result *= n
        n -= 1
    return result

print(factorial_rec(10), factorial_loop(10))

In [ ]:
def factorial_acc(n, result=1):
    if n == 0:
        return result
    return factorial_acc(n - 1, result * n)

print(factorial_acc(10))

In [ ]:
def nested_sum_rec(items):
    return sum(nested_sum_rec(x) if isinstance(x, list) else x for x in items)

def nested_sum_loop(items):
    total = 0
    todo = [items]
    while todo:
        current = todo.pop()
        for x in current:
            if isinstance(x, list):
                todo.append(x)
            else:
                total += x
    return total

data = [1, [2, 3], [4, [5, [6]]], 7]
print(nested_sum_rec(data), nested_sum_loop(data))

In [ ]:
deep = []
for _ in range(3000):
    deep = [deep, 1]

def nested_sum_loop(items):
    total = 0
    todo = [items]
    while todo:
        for x in todo.pop():
            if isinstance(x, list):
                todo.append(x)
            else:
                total += x
    return total

print(nested_sum_loop(deep))

> **Key idea.** Any recursion can be replaced by a loop and an explicit stack of work. It's more code, but it has no depth limit. For straight-line recursion, a simple loop with a variable is enough.

In [ ]:
import time

def total_rec(items):
    return 0 if not items else items[0] + total_rec(items[1:])

def total_loop(items):
    total = 0
    for x in items:
        total += x
    return total

data = list(range(900))
for f in (total_rec, total_loop, sum):
    start = time.perf_counter()
    for _ in range(100):
        f(data)
    print(f"{f.__name__:<10} {time.perf_counter() - start:.4f} s")

### Exercise: Recursion to loop

Rewrite `digit_count` with a `while` loop, no recursion.

```
def digit_count(n):
    if n < 10:
        return 1
    return 1 + digit_count(n // 10)
```

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert [digit_count(n) for n in (0, 9, 10, 2026, 10 ** 20)] == [1, 1, 2, 4, 21]\n    assert source.count(\"digit_count(\") == 1 and \"while\" in source")

### Exercise: Loop to recursion

Rewrite `count_upper` recursively, with no loops.

```
def count_upper(text):
    n = 0
    for ch in text:
        if ch.isupper():
            n += 1
    return n
```

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert count_upper(\"Python Atlas\") == 2 and count_upper(\"\") == 0 and count_upper(\"ABC\") == 3\n    assert \"for \" not in source and \"while \" not in source")

### Exercise: With an accumulator

Write `reverse_acc(text, done="")` recursively, so the reversed text is built up in the accumulator `done` on the way down, and returned complete from the base case. Nothing should happen after the recursive call.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert reverse_acc(\"stressed\") == \"desserts\" and reverse_acc(\"\") == \"\" and reverse_acc(\"a\") == \"a\"\n    assert \"for \" not in source and \"while \" not in source and \"[::-1]\" not in source\n    assert \"return reverse_acc(\" in source, \"return the recursive call's result directly\"")

### Exercise: Flatten with a stack

Write `flatten(items)` with a loop and an explicit stack, no recursion, keeping the items **in their original order**. It must handle 5,000 levels of nesting.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"order and depth\"\n    assert flatten([1, [2, 3], [4, [5, [6]]], 7]) == [1, 2, 3, 4, 5, 6, 7]\n    assert flatten([]) == [] and flatten([[], [[]]]) == []\n    deep = [9]\n    for _ in range(5000):\n        deep = [deep]\n    assert flatten(deep) == [9]\n    assert source.count(\"flatten(\") == 1, \"no recursion\"")

### Exercise: Every file, with a stack

A folder is nested dictionaries (a file's value is its size). Write `count_files(folder)` that counts every file at any depth, using a loop and a stack of folders still to look in, no recursion.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    f = {\"a.txt\": 1, \"docs\": {\"b.txt\": 2, \"old\": {\"c.txt\": 3, \"empty\": {}}}, \"d.txt\": 4}\n    assert count_files(f) == 4 and count_files({}) == 0\n    deep = {\"x.txt\": 1}\n    for i in range(3000):\n        deep = {f\"f{i}\": deep, \"y\": 1}\n    assert count_files(deep) == 3001\n    assert source.count(\"count_files(\") == 1")

### Exercise: Which would you use?

For each task, set the variable to `"recursion"` or `"loop"`: the better choice in Python, by today's rules.

In [ ]:
a = ___   # add up the numbers in a list of a million readings
b = ___   # count every comment in a discussion where replies can have replies, to any depth (rarely more than 20 levels)
c = ___   # find the first line of a log file that contains "ERROR"
d = ___   # list every file in a folder and all the folders inside it

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert (a, b, c, d) == (\"loop\", \"recursion\", \"loop\", \"recursion\")")

### Bug hunt: A stack that never empties

`total_nested` uses a stack, but it never finishes: it keeps processing the same list. Fix it.

In [ ]:
def total_nested(items):
    total = 0
    todo = [items]
    while todo:
        current = todo[-1]
        for x in current:
            if isinstance(x, list):
                todo.append(x)
            else:
                total += x
    return total

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert total_nested([1, [2, [3]]]) == 6 and total_nested([]) == 0")

### Puzzle: Power, as a loop

Arrange the lines to turn the recursive `power(x, n)` into a loop. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them

Work these out without running them.

In [ ]:
a = ___   # factorial_acc(4)  (from today)
b = ___   # how many frames does factorial_acc(4) have on the stack at its deepest, including the first?
c = ___   # with todo = [[1, [2]], [3]], todo.pop() gives...
d = ___   # True or False: Python turns tail calls into loops

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert a == 24 and b == 5 and c == [3] and d is False")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: both ways

A **binary representation as a list of bits** can be produced recursively or with a loop. Write both, and check them against each other:

- `bits_rec(n)`: the list of binary digits of a whole number n of 0 or more, most significant first, recursively (`bits_rec(6)` is `[1, 1, 0]`, `bits_rec(0)` is `[0]`);
- `bits_loop(n)`: the same with a `while` loop and no recursion;
- `from_bits(bits)`: the number a list of bits represents, with a loop (each step: value = value × 2 + bit). This is **Horner's method**, and it's how computers read numbers in any base.

Then check that `from_bits(bits_rec(n)) == n` and `bits_rec(n) == bits_loop(n)` for every n up to 1,000.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"all three\"\n    assert bits_rec(6) == [1, 1, 0] and bits_rec(0) == [0] and bits_rec(1) == [1]\n    assert bits_loop(6) == [1, 1, 0] and bits_loop(0) == [0]\n    assert from_bits([1, 1, 0]) == 6 and from_bits([0]) == 0 and from_bits([]) == 0\n    for n in range(1001):\n        assert bits_rec(n) == bits_loop(n) == [int(b) for b in bin(n)[2:]], n\n        assert from_bits(bits_loop(n)) == n\ndef test_shapes():\n    \"recursive and loop versions\"\n    body_rec = source[source.index(\"def bits_rec\"):source.index(\"def\", source.index(\"def bits_rec\") + 1)]\n    assert \"bits_rec(\" in body_rec[10:] and \"while\" not in body_rec and \"for \" not in body_rec")

## Warm-up

Same answer, two shapes. Now try `list(range(5000))` in your head: the loop is fine, but the recursive version needs 5,000 frames waiting at once, five times Python's limit. Both are correct; only one is suitable.

## Straight-line recursion is a loop

When a function makes **one** recursive call, on the "rest" of the problem, the calls form a straight line, not a tree. Straight-line recursion can always be rewritten as a loop. Compare:

The parts match up:

| Recursive | Loop |
|---|---|
| the base case | the loop's stopping condition |
| the recursive call on a smaller problem | one more time round the loop, with smaller values |
| combining on the way back up | a variable updated each time round |

The loop has to keep the partial result in a variable (`result`), because there are no frames waiting to combine values on the way back up.

## Accumulators

You can write the recursion the loop's way too: pass the partial result **down** as an extra parameter, called an **accumulator**, so that the answer is complete when the base case is reached:

Now nothing happens after the recursive call: its result is returned as it is. A call like that is a **tail call**. Some languages turn tail calls into loops automatically, so they use no extra stack. **Python doesn't**, on purpose: Guido van Rossum, Python's creator, preferred tracebacks that show every call. So in Python an accumulator doesn't avoid the recursion limit; its value is that it shows you how to write the loop.

## An explicit stack

Recursion that makes **several** calls, or works on nested data, isn't a straight line. To write it as a loop, you do what Python does for you: keep a **stack of work still to do**. Take a piece of work off the stack; if it's simple, deal with it; if it contains more work, push that onto the stack.

`todo` holds the lists still to be added up. The loop ends when there's nothing left to do. Because it's an ordinary list, not frames on the call stack, it has no depth limit. Here's a list nested 3,000 levels deep, which the recursive version can't handle (it would need 3,000 frames at once):

The order may differ: a stack takes the **most recently added** work first. For a sum that doesn't matter. When order matters (flattening, say), push the sub-items in reverse, or use the recursive version.

## Costs

Recursion in Python costs more than a loop: each call creates a frame, and `seq[1:]` copies the rest of the sequence each time. Let's measure:

The recursive version is many times slower, mostly because of the copying (each call copies nearly the whole list, so the total copying grows with the **square** of the length). The built-in `sum` is fastest of all. For a list, the loop is the right tool.

None of that applies to problems where recursion fits naturally, like nested data or the trees of calls you'll meet next week. There, the recursive version is usually just as fast, and much clearer.

## Choosing

| Use recursion when... | Use a loop when... |
|---|---|
| the data is nested or tree-shaped (folders, JSON, nested lists) | the data is a straight sequence |
| the problem splits into smaller copies of itself (next week) | each step just moves one item along |
| the recursive version is much clearer | the input can be long (thousands of levels) |
| the depth stays small (tens or hundreds of levels) | speed matters and a simple loop does it |

Clarity is the deciding factor more often than speed. A recursive `flatten` is five obvious lines; the stack version takes more thought to get right. A recursive `total` is a curiosity; the loop is what anyone would write.

## Practice

## Quiz

## Daily challenge

## Before you go

Take two recursive functions from this week and rewrite each as a loop; take two loops from earlier weeks and rewrite each recursively. For each pair, decide which version you'd keep, and why. Tomorrow is the Forge: recursion put to work on puzzles and data.